# Instance-Based Learning
## A Detailed Guide: k-Nearest Neighbours & Locally Weighted Regression

**References:**
- Tom Mitchell, *Machine Learning*, Chapter 8 — Instance-Based Learning
- Lecture 13 notes (CS456 — NISER, Spring 2023)
- Peter Harrington, *Machine Learning in Action*, Chapter 2

**Datasets:** Iris · Wine · Digits (scikit-learn)

---

This notebook is structured in two major parts:

| Part | Model | Module |
|------|-------|--------|
| **Part I** | k-Nearest Neighbours (k-NN) | `knn_applications.py` |
| **Part II** | Locally Weighted Regression (LWR) | `weighted_regression.py` |

Each part contains **Theory** (concepts, formulations, pros/cons) followed by **Applications** on three real datasets.

---
# PART I — k-Nearest Neighbours (k-NN)
---

## 1. Theory

### 1.1 Instance-Based (Lazy) Learning

Instance-based learning methods defer all generalisation until a new query is received (Mitchell Ch.8).  
Instead of building a global model during training, the learner simply **stores** all training examples and constructs a **local approximation** only when needed.

Advantages:
- Can represent extremely complex target functions (different hypothesis per query).
- No information is thrown away during training.

Disadvantages:
- Classification is expensive — must scan all stored examples.
- High memory usage.
- Susceptible to the **curse of dimensionality**.

### 1.2 k-NN Algorithm

Given a query $x_q$ and training set $D = \{(x_i, y_i)\}$:

1. Compute distance $d(x_q, x_i)$ for every stored example.
2. Sort distances in ascending order.
3. Select the $k$ nearest neighbours $N_k(x_q)$.
4. **Classification:** return the majority class   $\hat{y} = \arg\max_v \sum_{(x_i,y_i) \in N_k} \delta(v, y_i)$
5. **Regression:** return the mean  $\hat{f}(x_q) = \frac{1}{k} \sum_{(x_i,y_i) \in N_k} y_i$

### 1.3 Distance Metrics

| Metric | Formula | Notes |
|--------|---------|-------|
| **Euclidean** (L₂) | $d = \sqrt{\sum_i (x_i - y_i)^2}$ | Default; sensitive to scaling |
| **Manhattan** (L₁) | $d = \sum_i |x_i - y_i|$ | More robust in high-d |
| **Minkowski** (Lₚ) | $d = \left(\sum_i |x_i - y_i|^p\right)^{1/p}$ | Generalises L₁ and L₂ |
| **Chebyshev** (L∞) | $d = \max_i |x_i - y_i|$ | Only largest difference matters |

### 1.4 Effect of k

| k | Bias | Variance | Boundary | Risk |
|---|------|----------|----------|------|
| 1 | Low | High | Jagged (Voronoi) | Overfitting |
| 3–5 | ↑ | ↓ | Complex but smoother | Slight overfitting |
| 7–15 | Medium | Medium | **Best generalisation** | — |
| ≫√n | High | Low | Nearly linear | Underfitting |

**Rule of thumb:** $k \approx \sqrt{n}$, then cross-validate.

### 1.5 Distance-Weighted k-NN

Rather than treating all $k$ neighbours equally, weight each by the inverse square of its distance (Mitchell §8.2.1):

$$w_i = \frac{1}{d(x_q, x_i)^2}$$

$$\hat{y} = \arg\max_v \sum_{(x_i,y_i) \in N_k} w_i \cdot \delta(v, y_i)$$

Benefits: reduces sensitivity to the choice of $k$; even with $k = n$, only nearby points contribute.

### 1.6 Curse of Dimensionality

In high-dimensional spaces:
- Distances **concentrate** — all points appear equally far.
- Data needed grows **exponentially** with dimensionality.
- Irrelevant features add noise to distance → degraded performance.

Mitigations: PCA, feature selection, feature weighting, use Manhattan distance.

## 2. Theory Display (detailed terminal printout)

In [ ]:
import knn_applications

# Display all k-NN theory sections
knn_applications.display_all_theory()

## 3. Application — Dataset 1: Iris

The **Iris** dataset (150 samples, 4 features, 3 classes) is the classic ML benchmark.  
We analyse: accuracy vs $k$, distance metric comparison, uniform vs weighted, and decision boundaries.

In [ ]:
from sklearn.datasets import load_iris
knn_applications.demo_dataset(load_iris, "Iris")

### 3.1 Decision Boundary Visualisation (Iris, 2D)

Below we plot k-NN decision boundaries for $k = 1, 3, 7, 15$ using petal length and petal width.  
Notice how the boundary smooths out as $k$ increases.

In [ ]:
knn_applications.plot_decision_boundaries_iris()

## 4. Application — Dataset 2: Wine

The **Wine** dataset (178 samples, 13 features, 3 classes) tests k-NN in a higher-dimensional setting.  
Feature scaling is critical here because features have very different ranges.

In [ ]:
from sklearn.datasets import load_wine
knn_applications.demo_dataset(load_wine, "Wine")

## 5. Application — Dataset 3: Digits

The **Digits** dataset (1797 samples, 64 features = 8×8 pixel images, 10 classes) mirrors the handwriting recognition problem from Lecture 13.  
This is the most challenging dataset: high-dimensional, 10-class classification.

In [ ]:
from sklearn.datasets import load_digits
knn_applications.demo_dataset(load_digits, "Digits")

---
# PART II — Locally Weighted Regression (LWR)
---

## 6. Theory

### 6.1 What is Locally Weighted Regression?

LWR (Mitchell Ch.8 §8.3) generalises k-NN regression by fitting a **local linear model** around each query point $x_q$, rather than just averaging neighbour values.

**Key steps:**
1. Assign a weight to each training example: $w_i = \exp\left(-\frac{\|x_q - x_i\|^2}{2\tau^2}\right)$  (Gaussian kernel)
2. Solve weighted least squares: $\theta^* = (X^T W X)^{-1} X^T W y$
3. Predict: $\hat{y}_q = \theta^{*T} x_q$

### 6.2 Bandwidth Parameter τ

| τ | Effect |
|---|--------|
| Very small (0.01–0.1) | Only closest points matter → overfits, wiggly |
| Medium (0.5–2.0) | Good balance — captures trends, smooths noise |
| Very large (10+) | All points weighted equally → global linear regression |

τ is analogous to $k$ in k-NN: controls the bias-variance trade-off.

### 6.3 LWR vs k-NN Regression

| Aspect | k-NN Regression | LWR |
|--------|----------------|-----|
| Local model | None (just mean) | Linear / polynomial |
| Smoothness | Piecewise constant | Piecewise linear |
| Hyper-param | k | τ |
| Fit quality | Step-like | Smooth curve |
| Cost/query | O(n) | O(n·d²) |

## 7. Theory Display (detailed terminal printout)

In [ ]:
import weighted_regression

# Display all LWR theory sections
weighted_regression.display_all_theory()

## 8. Application — Demo 1: 1D Synthetic Data

A 1D non-linear function with noise is the best setting to **visualise** the effect of τ.  
We also compare LWR against k-NN regression and global linear regression.

In [ ]:
weighted_regression.demo_1d_synthetic()

## 9. Application — Demo 2: Iris (Regression)

We adapt the Iris dataset for regression: **predict petal width** from sepal length, sepal width, and petal length.  
This lets us compare LWR, k-NN regression, and global LR on a real multi-feature dataset.

In [ ]:
weighted_regression.demo_iris_regression()

## 10. Application — Demo 3: Wine (Regression)

We predict **alcohol content** from the remaining 12 features of the Wine dataset.  
This tests LWR in a higher-dimensional space (12 features).

In [ ]:
weighted_regression.demo_wine_regression()

---
## Summary & Key Takeaways

| | k-NN | LWR |
|---|------|-----|
| **Type** | Classification & Regression | Regression |
| **Hyper-parameter** | k (number of neighbours) | τ (bandwidth) |
| **Local model** | None (majority vote / mean) | Weighted linear fit |
| **Bias-variance** | k controls trade-off | τ controls trade-off |
| **Fit smoothness** | Piecewise constant | Piecewise linear (smooth) |
| **Best for** | Classification, simple regression | Smooth non-linear regression |
| **Key weakness** | Curse of dimensionality | Computational cost O(n·d²)/query |

**Both are instance-based (lazy) learners:**
- No training phase — all computation at query time.
- Must store the full training set in memory.
- Feature scaling is critical for both.
- Performance degrades in very high dimensions.

---
*Notebook authored as part of CS456 — Machine Learning coursework.*